# Préparation du dataset analytique

## Objectif

Cette étape vise à transformer les mesures énergétiques brutes issues de l'API EnerVision en un jeu de données structuré, nettoyé et adapté à l'entraînement de modèles de Machine Learning.

L'objectif métier est de prévoir, pour chacun des sept sites, l'évolution de la puissance électrique sur les prochaines 24 heures, avec une granularité horaire.

## Méthodologie

La préparation repose sur les données hebdomadaires collectées lors de la phase d'exploration.

Les traitements porteront notamment sur :

- La validation de la structure et de la cohérence des données.
- L'identification des doublons et des valeurs manquantes.
- La prise en compte des indicateurs de qualité.
- L'agrégation des observations à une granularité horaire.
- La construction de variables explicatives adaptées à la prévision.

Les données brutes seront conservées sans modification. Le dataset analytique sera exporté séparément afin de garantir la traçabilité des transformations.

**Principe directeur :** les règles de nettoyage doivent être définies à partir des anomalies observées et ne doivent pas introduire d'informations indisponibles au moment d'une prédiction.

## A — Chargement des données hebdomadaires

### Préparation des données sources

Cette étape repose sur les mesures énergétiques collectées depuis l'API EnerVision pour les sept sites étudiés.

Afin de garantir la reproductibilité des traitements, la collecte des données est réalisée indépendamment du notebook, à l'aide du script `scripts/collect_readings.py`.

**Avant d'exécuter ce notebook**, lancer la commande suivante depuis le répertoire `ml/` :

```powershell
python scripts/collect_readings.py --days 7
```

Les fichiers JSON générés sont enregistrés dans le répertoire :

```text
ml/data/raw/7d/
```

Chaque fichier contient les mesures d'un site ainsi que les métadonnées de collecte : date de récupération, début et fin de la période étudiée.

### Périmètre de l'analyse

Les données hebdomadaires constituent un premier jeu de travail permettant de développer et de valider notre pipeline de préparation analytique.

Elles serviront notamment à définir les règles de nettoyage, contrôler la cohérence temporelle et construire les premières variables explicatives.

**Limite identifiée :** cet historique d'une semaine est insuffisant pour garantir un entraînement représentatif d'un modèle de prévision à 24 heures. Un historique continu plus important sera nécessaire lors de la phase d'expérimentation.

### Reproductibilité

Le notebook exploite les fichiers JSON préalablement collectés et n'effectue aucun appel direct à l'API.

Les données brutes sont conservées sans modification. Les résultats des transformations seront enregistrés séparément dans `data/processed/`.

A.1 — Chargement des données brutes

In [105]:
import json
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/raw/7d")

files = sorted(DATA_DIR.glob("SITE*.json"))

if not files:
    raise FileNotFoundError(
        f"Aucun fichier JSON trouvé dans {DATA_DIR.resolve()}. "
        "Exécuter le script de collecte au préalable."
    )

print(f"Fichiers détectés : {len(files)}")

collections = []

for path in files:
    with path.open(encoding="utf-8") as file:
        payload = json.load(file)

    collections.append(
        {
            "filename": path.name,
            "site_id": payload["site_id"],
            "collected_at": payload["collected_at"],
            "period_start": payload["period_start"],
            "period_end": payload["period_end"],
            "readings_count": len(payload["readings"]),
            "readings": payload["readings"],
        }
    )

df_collections = pd.DataFrame(collections)

display(df_collections.drop(columns=["readings"]))

Fichiers détectés : 7


,filename,site_id,collected_at,period_start,period_end,readings_count
0,SITE001_20261023_100003.json,SITE001,2026-10-23T10:00:03.252226+00:00,2026-10-16T09:55:03.252226+00:00,2026-10-23T09:55:03.252226+00:00,1000
1,SITE002_20261023_100003.json,SITE002,2026-10-23T10:00:03.252226+00:00,2026-10-16T09:55:03.252226+00:00,2026-10-23T09:55:03.252226+00:00,1000
2,SITE003_20261023_100003.json,SITE003,2026-10-23T10:00:03.252226+00:00,2026-10-16T09:55:03.252226+00:00,2026-10-23T09:55:03.252226+00:00,1000
3,SITE004_20261023_100003.json,SITE004,2026-10-23T10:00:03.252226+00:00,2026-10-16T09:55:03.252226+00:00,2026-10-23T09:55:03.252226+00:00,1000
4,SITE005_20261023_100003.json,SITE005,2026-10-23T10:00:03.252226+00:00,2026-10-16T09:55:03.252226+00:00,2026-10-23T09:55:03.252226+00:00,1000
5,SITE006_20261023_100003.json,SITE006,2026-10-23T10:00:03.252226+00:00,2026-10-16T09:55:03.252226+00:00,2026-10-23T09:55:03.252226+00:00,1000
6,SITE007_20261023_100003.json,SITE007,2026-10-23T10:00:03.252226+00:00,2026-10-16T09:55:03.252226+00:00,2026-10-23T09:55:03.252226+00:00,1000


A.2 — Contrôle des campagnes de collecte

In [106]:
# Conversion des métadonnées temporelles
for column in ["collected_at", "period_start", "period_end"]:
    df_collections[column] = pd.to_datetime(
        df_collections[column],
        utc=True,
        format="ISO8601",
    )

print("Nombre de fichiers :", len(df_collections))
print("Nombre de sites :", df_collections["site_id"].nunique())

print("\nNombre de fichiers par site :")
display(df_collections.groupby("site_id").size())

print("\nPériodes de collecte distinctes :")
display(
    df_collections[["period_start", "period_end"]].drop_duplicates().sort_values("period_start")
)

print("\nDates de collecte :")
display(df_collections[["filename", "site_id", "collected_at"]].sort_values("collected_at"))

Nombre de fichiers : 7
Nombre de sites : 7

Nombre de fichiers par site :


site_id
SITE001    1
SITE002    1
SITE003    1
SITE004    1
SITE005    1
SITE006    1
SITE007    1
dtype: int64


Périodes de collecte distinctes :


,period_start,period_end
0,2026-10-16 09:55:03.252226+00:00,2026-10-23 09:55:03.252226+00:00



Dates de collecte :


,filename,site_id,collected_at
0,SITE001_20261023_100003.json,SITE001,2026-10-23 10:00:03.252226+00:00
1,SITE002_20261023_100003.json,SITE002,2026-10-23 10:00:03.252226+00:00
2,SITE003_20261023_100003.json,SITE003,2026-10-23 10:00:03.252226+00:00
3,SITE004_20261023_100003.json,SITE004,2026-10-23 10:00:03.252226+00:00
4,SITE005_20261023_100003.json,SITE005,2026-10-23 10:00:03.252226+00:00
5,SITE006_20261023_100003.json,SITE006,2026-10-23 10:00:03.252226+00:00
6,SITE007_20261023_100003.json,SITE007,2026-10-23 10:00:03.252226+00:00


A.3 — Validation avant concaténation

In [107]:
expected_sites = {f"SITE{i:03d}" for i in range(1, 8)}

actual_sites = set(df_collections["site_id"])

assert actual_sites == expected_sites, (
    f"Sites manquants ou inattendus : {expected_sites.symmetric_difference(actual_sites)}"
)

assert len(df_collections) == 7, (
    "Le dossier doit contenir exactement un fichier par site. "
    "Vérifier les campagnes de collecte présentes."
)

assert df_collections["period_start"].nunique() == 1, (
    "Les fichiers ne partagent pas la même date de début."
)

assert df_collections["period_end"].nunique() == 1, (
    "Les fichiers ne partagent pas la même date de fin."
)

assert (df_collections["readings_count"] > 0).all(), (
    "Au moins un fichier ne contient aucune mesure."
)

print("Validation des fichiers sources réussie.")

Validation des fichiers sources réussie.


A.4 — Construction du DataFrame brut

In [108]:
records = []

for collection in df_collections.itertuples():
    records.extend(collection.readings)

df_raw = pd.DataFrame(records)

df_raw["timestamp"] = pd.to_datetime(
    df_raw["timestamp"],
    utc=True,
    format="ISO8601",
    errors="coerce",
)

print(f"Observations chargées : {len(df_raw):,}")
print(f"Sites distincts : {df_raw['site_id'].nunique()}")
print(f"Colonnes : {len(df_raw.columns)}")

display(df_raw.head())
display(df_raw.dtypes.to_frame("Type"))

Observations chargées : 7,000
Sites distincts : 7
Colonnes : 12


,timestamp,site_id,site_type,consumption_kw,consumption_kwh,voltage_v,current_a,power_factor,temperature_celsius,humidity_percent,null_reasons,data_quality
0,2026-10-16 09:55:03.252226+00:00,SITE001,office,124.11,124.11,392.1,192.36,0.919,10.1,40.5,[],good
1,2026-10-16 10:05:08.052226+00:00,SITE001,office,181.80,181.80,401.0,275.53,0.959,7.6,46.5,[],good
2,2026-10-16 10:15:12.852226+00:00,SITE001,office,NaN,NaN,413.5,249.69,0.936,7.3,37.1,[consumption_sensor_failure],partial
3,2026-10-16 10:25:17.652226+00:00,SITE001,office,151.87,151.87,399.5,231.06,0.921,11.4,40.2,[],good
4,2026-10-16 10:35:22.452226+00:00,SITE001,office,180.89,180.89,397.0,276.93,0.906,13.4,NaN,[humidity_sensor_failure],partial


,Type
timestamp,"datetime64[us, UTC]"
site_id,str
site_type,str
consumption_kw,float64
consumption_kwh,float64
voltage_v,float64
current_a,float64
power_factor,float64
temperature_celsius,float64
humidity_percent,float64


## B — Contrôle qualité des données

Avant de procéder au nettoyage, nous réalisons un audit des observations afin d'identifier les anomalies susceptibles d'affecter la qualité du futur modèle prédictif.

Cet audit porte sur quatre dimensions :

- **Complétude :** identification et quantification des valeurs manquantes.
- **Unicité :** recherche des observations dupliquées.
- **Cohérence :** contrôle des valeurs numériques et des indicateurs de qualité.
- **Continuité temporelle :** analyse de la fréquence et de la répartition des mesures.

Les résultats permettront de définir des règles de nettoyage explicites et justifiées, sans altérer les données sources.

A - Analyse des valeurs manquantes

In [109]:
missing = pd.DataFrame(
    {
        "missing_count": df_raw.isna().sum(),
        "missing_percent": (df_raw.isna().mean() * 100).round(2),
    }
)

display(
    missing.sort_values(
        "missing_percent",
        ascending=False,
    )
)

,missing_count,missing_percent
temperature_celsius,609,8.70
humidity_percent,531,7.59
consumption_kw,429,6.13
consumption_kwh,429,6.13
power_factor,405,5.79
current_a,405,5.79
voltage_v,405,5.79
timestamp,0,0.00
site_type,0,0.00
site_id,0,0.00


In [110]:
numeric_columns = [
    "consumption_kw",
    "temperature_celsius",
    "humidity_percent",
    "voltage_v",
    "current_a",
    "power_factor",
]

missing_by_site = (
    df_raw.groupby("site_id")[numeric_columns]
    .apply(lambda group: group.isna().mean() * 100)
    .round(2)
)

display(missing_by_site)

,consumption_kw,temperature_celsius,humidity_percent,voltage_v,current_a,power_factor
site_id,,,,,,
SITE001,7.4,9.7,9.1,7.1,7.1,7.1
SITE002,6.5,9.6,7.5,6.1,6.1,6.1
SITE003,6.5,9.5,8.2,7.0,7.0,7.0
SITE004,5.7,7.7,8.4,4.4,4.4,4.4
SITE005,5.5,9.0,7.7,4.9,4.9,4.9
SITE006,5.6,7.1,6.0,5.3,5.3,5.3
SITE007,5.7,8.3,6.2,5.7,5.7,5.7


B - Analyse des doublons

In [111]:
duplicate_count = df_raw.duplicated(subset=["site_id", "timestamp"]).sum()

print(f"Doublons site / timestamp : {duplicate_count}")

if duplicate_count > 0:
    display(
        df_raw[
            df_raw.duplicated(
                subset=["site_id", "timestamp"],
                keep=False,
            )
        ]
        .sort_values(["site_id", "timestamp"])
        .head(20)
    )

Doublons site / timestamp : 0


C - Distribution des indicateurs de qualité

In [112]:
display(
    df_raw["data_quality"]
    .value_counts(dropna=False)
    .rename_axis("Qualité")
    .reset_index(name="Observations")
)

display(
    pd.crosstab(
        df_raw["site_id"],
        df_raw["data_quality"],
        dropna=False,
    )
)

,Qualité,Observations
0,good,5534
1,partial,1230
2,critical,134
3,degraded,102


data_quality,critical,degraded,good,partial
site_id,,,,
SITE001,25,22,766,187
SITE002,20,11,774,195
SITE003,28,14,786,172
SITE004,15,10,793,182
SITE005,16,16,794,174
SITE006,13,11,811,165
SITE007,17,18,810,155


D - Vérification des plages de valeurs

In [113]:
display(df_raw[numeric_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
consumption_kw,6571.0,393.489589,284.774540,47.33,127.785,327.96,678.425,1054.22
temperature_celsius,6391.0,5.030199,4.110569,-5.90,1.700,5.00,8.300,16.40
humidity_percent,6469.0,42.797032,4.985182,30.00,39.400,42.70,46.200,60.40
voltage_v,6595.0,399.970553,4.991921,376.80,396.600,399.90,403.400,417.70
current_a,6595.0,596.698763,432.339699,70.99,194.690,498.49,1022.210,1610.68
power_factor,6595.0,0.920350,0.029274,0.85,0.900,0.92,0.941,0.98


In [114]:
anomalies = {
    "consommation_negative": (df_raw["consumption_kw"] < 0).sum(),
    "humidite_hors_plage": (
        ~df_raw["humidity_percent"].between(0, 100) & df_raw["humidity_percent"].notna()
    ).sum(),
    "facteur_puissance_hors_plage": (
        ~df_raw["power_factor"].between(0, 1) & df_raw["power_factor"].notna()
    ).sum(),
}

display(
    pd.DataFrame(
        anomalies.items(),
        columns=["Anomalie", "Occurrences"],
    )
)

,Anomalie,Occurrences
0,consommation_negative,0
1,humidite_hors_plage,0
2,facteur_puissance_hors_plage,0


E - Cohérence entre qualité et valeurs manquantes 

In [115]:
columns_to_check = [
    "consumption_kw",
    "temperature_celsius",
    "humidity_percent",
    "voltage_v",
    "current_a",
    "power_factor",
]

df_quality = df_raw.copy()

df_quality["missing_count"] = df_quality[columns_to_check].isna().sum(axis=1)

display(
    df_quality.groupby("data_quality").agg(
        observations=("missing_count", "size"),
        missing_mean=("missing_count", "mean"),
        missing_max=("missing_count", "max"),
        complete_rows=(
            "missing_count",
            lambda x: (x == 0).sum(),
        ),
    )
)

print("Absence simultanée des mesures électriques :")

electrical_columns = [
    "voltage_v",
    "current_a",
    "power_factor",
]

display(df_quality[electrical_columns].isna().value_counts().rename("observations").reset_index())

,observations,missing_mean,missing_max,complete_rows
data_quality,,,,
critical,134,6.000000,6,0
degraded,102,2.862745,5,0
good,5534,0.000000,0,5534
partial,1230,1.372358,3,0


Absence simultanée des mesures électriques :


,voltage_v,current_a,power_factor,observations
0,False,False,False,6595
1,True,True,True,405


F - Continuité temporelle

In [116]:
df_temporal = df_raw.sort_values(["site_id", "timestamp"]).copy()

df_temporal["interval_minutes"] = (
    df_temporal.groupby("site_id")["timestamp"].diff().dt.total_seconds() / 60
)

temporal_summary = df_temporal.groupby("site_id").agg(
    observations=("timestamp", "size"),
    first_timestamp=("timestamp", "min"),
    last_timestamp=("timestamp", "max"),
    median_interval_min=("interval_minutes", "median"),
    max_interval_min=("interval_minutes", "max"),
)

display(temporal_summary)

print("Distribution des intervalles :")

display(df_temporal["interval_minutes"].describe())

,observations,first_timestamp,last_timestamp,median_interval_min,max_interval_min
site_id,,,,,
SITE001,1000,2026-10-16 09:55:03.252226+00:00,2026-10-23 09:44:58.452226+00:00,10.08,10.08
SITE002,1000,2026-10-16 09:55:03.252226+00:00,2026-10-23 09:44:58.452226+00:00,10.08,10.08
SITE003,1000,2026-10-16 09:55:03.252226+00:00,2026-10-23 09:44:58.452226+00:00,10.08,10.08
SITE004,1000,2026-10-16 09:55:03.252226+00:00,2026-10-23 09:44:58.452226+00:00,10.08,10.08
SITE005,1000,2026-10-16 09:55:03.252226+00:00,2026-10-23 09:44:58.452226+00:00,10.08,10.08
SITE006,1000,2026-10-16 09:55:03.252226+00:00,2026-10-23 09:44:58.452226+00:00,10.08,10.08
SITE007,1000,2026-10-16 09:55:03.252226+00:00,2026-10-23 09:44:58.452226+00:00,10.08,10.08


Distribution des intervalles :


count    6993.00
mean       10.08
std         0.00
min        10.08
25%        10.08
50%        10.08
75%        10.08
max        10.08
Name: interval_minutes, dtype: float64

## C — Définition des règles de nettoyage

### Synthèse du contrôle qualité

L'analyse des 7 000 observations issues des sept sites EnerVision met en évidence une structure temporelle régulière, avec un intervalle constant d'environ 10,08 minutes entre deux mesures successives.

Aucun doublon temporel ni aucune anomalie physique n'a été identifié lors des contrôles réalisés.

En revanche, plusieurs variables présentent des valeurs manquantes, notamment la température (8,70 %), l'humidité (7,59 %) et la puissance électrique (6,13 %).

Les indicateurs `data_quality` permettent de distinguer les observations complètes des observations partiellement ou totalement incomplètes.

### Stratégie retenue

Le nettoyage repose sur les principes suivants :

- Préserver les données brutes sans modification.
- Écarter les observations dont les identifiants temporels ou les identifiants de site sont invalides.
- Éliminer les éventuels doublons après vérification.
- Ne pas effectuer d'imputation systématique des valeurs de consommation, afin de ne pas introduire de cibles artificielles dans l'entraînement.
- Conserver les valeurs manquantes des variables météorologiques à ce stade.
- Agréger les mesures à une granularité horaire.
- Calculer un indicateur de couverture pour chaque heure et chaque site.

Une heure sera initialement considérée comme exploitable lorsque sa couverture en mesures de puissance atteint au moins 50 % des observations attendues.

Cette stratégie vise à préserver la fiabilité des données tout en limitant la perte d'informations disponibles.

### Prévention des fuites de données

Les variables explicatives destinées à la prédiction devront être disponibles au moment où celle-ci est effectuée.

Les mesures électriques contemporaines de l'heure à prédire ne seront donc pas utilisées directement comme variables explicatives. Les données météorologiques futures nécessiteront également une source de prévisions adaptée.

A - Nettoyage structurel

In [117]:
df_clean = df_raw.copy()

# Suppression des identifiants invalides
df_clean = df_clean.dropna(subset=["site_id", "timestamp"])

# Suppression des éventuels doublons
df_clean = df_clean.drop_duplicates(
    subset=["site_id", "timestamp"],
    keep="first",
)

# Tri chronologique
df_clean = df_clean.sort_values(["site_id", "timestamp"]).reset_index(drop=True)

print(f"Observations initiales : {len(df_raw)}")
print(f"Observations conservées : {len(df_clean)}")
print(f"Observations supprimées : {len(df_raw) - len(df_clean)}")

display(df_clean.head())

Observations initiales : 7000
Observations conservées : 7000
Observations supprimées : 0


,timestamp,site_id,site_type,consumption_kw,consumption_kwh,voltage_v,current_a,power_factor,temperature_celsius,humidity_percent,null_reasons,data_quality
0,2026-10-16 09:55:03.252226+00:00,SITE001,office,124.11,124.11,392.1,192.36,0.919,10.1,40.5,[],good
1,2026-10-16 10:05:08.052226+00:00,SITE001,office,181.80,181.80,401.0,275.53,0.959,7.6,46.5,[],good
2,2026-10-16 10:15:12.852226+00:00,SITE001,office,NaN,NaN,413.5,249.69,0.936,7.3,37.1,[consumption_sensor_failure],partial
3,2026-10-16 10:25:17.652226+00:00,SITE001,office,151.87,151.87,399.5,231.06,0.921,11.4,40.2,[],good
4,2026-10-16 10:35:22.452226+00:00,SITE001,office,180.89,180.89,397.0,276.93,0.906,13.4,NaN,[humidity_sensor_failure],partial


B - Aggrégation horaire des données
- Passage de 7000 observations à une table contenant une ligne par site et par heure

B1 - Construction table horaire

In [118]:
# Création de la période horaire
df_clean["timestamp_hour"] = df_clean["timestamp"].dt.floor("h")

# Agrégation par site et par heure
df_hourly = df_clean.groupby(
    ["site_id", "site_type", "timestamp_hour"],
    as_index=False,
).agg(
    consumption_kw=("consumption_kw", "mean"),
    temperature_celsius=("temperature_celsius", "mean"),
    humidity_percent=("humidity_percent", "mean"),
    observation_count=("timestamp", "size"),
    valid_consumption_count=("consumption_kw", "count"),
    good_count=("data_quality", lambda x: (x == "good").sum()),
)

display(df_hourly.head(10))

,site_id,site_type,timestamp_hour,consumption_kw,temperature_celsius,humidity_percent,observation_count,valid_consumption_count,good_count
0,SITE001,office,2026-10-16 09:00:00+00:00,124.110000,10.100000,40.500000,1,1,1
1,SITE001,office,2026-10-16 10:00:00+00:00,174.840000,9.383333,40.760000,6,5,4
2,SITE001,office,2026-10-16 11:00:00+00:00,166.120000,8.640000,44.640000,6,5,5
3,SITE001,office,2026-10-16 12:00:00+00:00,169.913333,9.400000,41.600000,6,6,4
4,SITE001,office,2026-10-16 13:00:00+00:00,175.741667,8.900000,41.133333,6,6,6
5,SITE001,office,2026-10-16 14:00:00+00:00,175.395000,9.450000,42.483333,6,4,4
6,SITE001,office,2026-10-16 15:00:00+00:00,163.173333,7.466667,43.640000,6,6,5
7,SITE001,office,2026-10-16 16:00:00+00:00,166.830000,9.200000,43.533333,6,6,6
8,SITE001,office,2026-10-16 17:00:00+00:00,170.803333,6.866667,47.640000,6,6,5
9,SITE001,office,2026-10-16 18:00:00+00:00,128.626667,4.916667,40.900000,6,6,5


In [119]:
# Complétude des mesures enregistrées
df_hourly["completeness_rate"] = (
    df_hourly["valid_consumption_count"] / df_hourly["observation_count"]
)

# Seuil minimal de trois mesures valides par heure
MIN_VALID_MEASUREMENTS = 3

df_hourly["is_valid_hour"] = df_hourly["valid_consumption_count"] >= MIN_VALID_MEASUREMENTS

print("Répartition des heures :")

display(df_hourly["is_valid_hour"].value_counts())

print("\nHeures invalides :")

display(
    df_hourly.loc[
        ~df_hourly["is_valid_hour"],
        [
            "site_id",
            "timestamp_hour",
            "observation_count",
            "valid_consumption_count",
            "completeness_rate",
        ],
    ]
)

Répartition des heures :


is_valid_hour
True     1176
False       7
Name: count, dtype: int64


Heures invalides :


,site_id,timestamp_hour,observation_count,valid_consumption_count,completeness_rate
0,SITE001,2026-10-16 09:00:00+00:00,1,1,1.0
169,SITE002,2026-10-16 09:00:00+00:00,1,1,1.0
338,SITE003,2026-10-16 09:00:00+00:00,1,1,1.0
507,SITE004,2026-10-16 09:00:00+00:00,1,1,1.0
676,SITE005,2026-10-16 09:00:00+00:00,1,1,1.0
845,SITE006,2026-10-16 09:00:00+00:00,1,1,1.0
1014,SITE007,2026-10-16 09:00:00+00:00,1,1,1.0


B2 - Vérification qualité agrégation

In [120]:
print("Nombre de lignes horaires :", len(df_hourly))

print("\nObservations par heure :")
display(df_hourly["observation_count"].value_counts().sort_index())

print("\nCouverture des consommations :")
display(df_hourly["completeness_rate"].describe())

print("\nHeures valides / invalides :")
display(df_hourly["is_valid_hour"].value_counts())

print("\nHeures invalides par site :")
display(
    pd.crosstab(
        df_hourly["site_id"],
        df_hourly["is_valid_hour"],
    )
)

df_hourly.loc[
    ~df_hourly["is_valid_hour"],
    "consumption_kw",
] = float("nan")

Nombre de lignes horaires : 1183

Observations par heure :


observation_count
1       7
5      63
6    1113
Name: count, dtype: int64


Couverture des consommations :


count    1183.000000
mean        0.939138
std         0.097872
min         0.500000
25%         0.833333
50%         1.000000
75%         1.000000
max         1.000000
Name: completeness_rate, dtype: float64


Heures valides / invalides :


is_valid_hour
True     1176
False       7
Name: count, dtype: int64


Heures invalides par site :


is_valid_hour,False,True
site_id,,
SITE001,1,168
SITE002,1,168
SITE003,1,168
SITE004,1,168
SITE005,1,168
SITE006,1,168
SITE007,1,168


C - Création des premières variables explicatives

C1 - Variables temporelles

In [121]:
df_hourly["hour"] = df_hourly["timestamp_hour"].dt.hour

df_hourly["day_of_week"] = df_hourly["timestamp_hour"].dt.dayofweek

df_hourly["month"] = df_hourly["timestamp_hour"].dt.month

df_hourly["is_weekend"] = (df_hourly["day_of_week"] >= 5).astype(int)

display(
    df_hourly[
        [
            "site_id",
            "timestamp_hour",
            "hour",
            "day_of_week",
            "is_weekend",
        ]
    ].head(15)
)

,site_id,timestamp_hour,hour,day_of_week,is_weekend
0,SITE001,2026-10-16 09:00:00+00:00,9,4,0
1,SITE001,2026-10-16 10:00:00+00:00,10,4,0
2,SITE001,2026-10-16 11:00:00+00:00,11,4,0
3,SITE001,2026-10-16 12:00:00+00:00,12,4,0
4,SITE001,2026-10-16 13:00:00+00:00,13,4,0
5,SITE001,2026-10-16 14:00:00+00:00,14,4,0
6,SITE001,2026-10-16 15:00:00+00:00,15,4,0
7,SITE001,2026-10-16 16:00:00+00:00,16,4,0
8,SITE001,2026-10-16 17:00:00+00:00,17,4,0
9,SITE001,2026-10-16 18:00:00+00:00,18,4,0


D - Adaptation des variables calendaires

In [122]:
# Référence technique : UTC
df_hourly["timestamp_local"] = df_hourly["timestamp_hour"].dt.tz_convert("Europe/Paris")

# Variables calendaires métier
df_hourly["hour"] = df_hourly["timestamp_local"].dt.hour
df_hourly["day_of_week"] = df_hourly["timestamp_local"].dt.dayofweek
df_hourly["month"] = df_hourly["timestamp_local"].dt.month

df_hourly["is_weekend"] = (df_hourly["day_of_week"] >= 5).astype(int)

E - Vérification de la continuité horaire

In [123]:
df_hourly = df_hourly.sort_values(["site_id", "timestamp_hour"]).reset_index(drop=True)

df_hourly["interval_hours"] = (
    df_hourly.groupby("site_id")["timestamp_hour"].diff().dt.total_seconds() / 3600
)

display(df_hourly.groupby("site_id")["interval_hours"].agg(["min", "median", "max"]))

print(
    "Intervalles supérieurs à une heure :",
    (df_hourly["interval_hours"] > 1).sum(),
)

print(
    "Doublons horaires :",
    df_hourly.duplicated(subset=["site_id", "timestamp_hour"]).sum(),
)

,min,median,max
site_id,,,
SITE001,1.0,1.0,1.0
SITE002,1.0,1.0,1.0
SITE003,1.0,1.0,1.0
SITE004,1.0,1.0,1.0
SITE005,1.0,1.0,1.0
SITE006,1.0,1.0,1.0
SITE007,1.0,1.0,1.0


Intervalles supérieurs à une heure : 0
Doublons horaires : 0


F - Création des variables historiques

In [124]:
df_hourly = df_hourly.sort_values(["site_id", "timestamp_hour"]).reset_index(drop=True)

grouped = df_hourly.groupby("site_id")["consumption_kw"]

# Consommations historiques
df_hourly["lag_1h"] = grouped.shift(1)
df_hourly["lag_24h"] = grouped.shift(24)

# Moyenne des 24 heures précédentes
df_hourly["rolling_mean_24h"] = df_hourly.groupby("site_id")["consumption_kw"].transform(
    lambda series: (
        series.shift(1)
        .rolling(
            window=24,
            min_periods=18,
        )
        .mean()
    )
)

display(
    df_hourly[
        [
            "site_id",
            "timestamp_hour",
            "consumption_kw",
            "lag_1h",
            "lag_24h",
            "rolling_mean_24h",
        ]
    ].iloc[20:30]
)

,site_id,timestamp_hour,consumption_kw,lag_1h,lag_24h,rolling_mean_24h
20,SITE001,2026-10-17 05:00:00+00:00,74.473333,73.404000,NaN,114.504895
21,SITE001,2026-10-17 06:00:00+00:00,72.952000,74.473333,NaN,112.503317
22,SITE001,2026-10-17 07:00:00+00:00,71.236000,72.952000,NaN,110.619921
23,SITE001,2026-10-17 08:00:00+00:00,74.890000,71.236000,NaN,108.829742
24,SITE001,2026-10-17 09:00:00+00:00,70.724000,74.890000,NaN,107.354101
25,SITE001,2026-10-17 10:00:00+00:00,76.056667,70.724000,174.840000,105.827847
26,SITE001,2026-10-17 11:00:00+00:00,75.016000,76.056667,166.120000,101.711875
27,SITE001,2026-10-17 12:00:00+00:00,73.342000,75.016000,169.913333,97.915875
28,SITE001,2026-10-17 13:00:00+00:00,74.875000,73.342000,175.741667,93.892069
29,SITE001,2026-10-17 14:00:00+00:00,75.404000,74.875000,175.395000,89.689292


G - Contrôle des variables historiques

In [125]:
lag_columns = [
    "lag_1h",
    "lag_24h",
    "rolling_mean_24h",
]

display(df_hourly[lag_columns].isna().sum().to_frame("missing_count"))

,missing_count
lag_1h,14
lag_24h,175
rolling_mean_24h,133


H - Construction des exemples de prévision

In [126]:
training_frames = []

df_hourly = df_hourly.sort_values(["site_id", "timestamp_hour"]).reset_index(drop=True)

for horizon in range(1, 25):
    df_horizon = df_hourly.copy()

    df_horizon["horizon"] = horizon

    # Instant auquel la prévision est effectuée
    df_horizon["forecast_origin"] = df_horizon["timestamp_hour"]

    # Heure que nous cherchons à prédire
    df_horizon["target_timestamp"] = df_horizon["forecast_origin"] + pd.Timedelta(hours=horizon)

    # Consommation future à prédire
    df_horizon["target_consumption_kw"] = df_horizon.groupby("site_id")["consumption_kw"].shift(
        -horizon
    )

    training_frames.append(df_horizon)

df_training = pd.concat(
    training_frames,
    ignore_index=True,
)

# Exclusion des exemples dont la cible est inconnue
df_training = df_training.dropna(subset=["target_consumption_kw"]).reset_index(drop=True)

print("Nombre d'exemples :", len(df_training))

display(
    df_training[
        [
            "site_id",
            "forecast_origin",
            "horizon",
            "target_timestamp",
            "target_consumption_kw",
            "lag_1h",
            "lag_24h",
            "rolling_mean_24h",
        ]
    ].head(10)
)

Nombre d'exemples : 26292


,site_id,forecast_origin,horizon,target_timestamp,target_consumption_kw,lag_1h,lag_24h,rolling_mean_24h
0,SITE001,2026-10-16 09:00:00+00:00,1,2026-10-16 10:00:00+00:00,174.840000,NaN,NaN,NaN
1,SITE001,2026-10-16 10:00:00+00:00,1,2026-10-16 11:00:00+00:00,166.120000,NaN,NaN,NaN
2,SITE001,2026-10-16 11:00:00+00:00,1,2026-10-16 12:00:00+00:00,169.913333,174.840000,NaN,NaN
3,SITE001,2026-10-16 12:00:00+00:00,1,2026-10-16 13:00:00+00:00,175.741667,166.120000,NaN,NaN
4,SITE001,2026-10-16 13:00:00+00:00,1,2026-10-16 14:00:00+00:00,175.395000,169.913333,NaN,NaN
5,SITE001,2026-10-16 14:00:00+00:00,1,2026-10-16 15:00:00+00:00,163.173333,175.741667,NaN,NaN
6,SITE001,2026-10-16 15:00:00+00:00,1,2026-10-16 16:00:00+00:00,166.830000,175.395000,NaN,NaN
7,SITE001,2026-10-16 16:00:00+00:00,1,2026-10-16 17:00:00+00:00,170.803333,163.173333,NaN,NaN
8,SITE001,2026-10-16 17:00:00+00:00,1,2026-10-16 18:00:00+00:00,128.626667,166.830000,NaN,NaN
9,SITE001,2026-10-16 18:00:00+00:00,1,2026-10-16 19:00:00+00:00,63.660000,170.803333,NaN,NaN


I - Compléter les variables calendaires 

In [127]:
# Conversion de l'heure cible dans le fuseau métier
target_local = df_training["target_timestamp"].dt.tz_convert("Europe/Paris")

# Variables calendaires de l'heure à prédire
df_training["target_hour"] = target_local.dt.hour

df_training["target_day_of_week"] = target_local.dt.dayofweek

df_training["target_month"] = target_local.dt.month

df_training["target_is_weekend"] = (df_training["target_day_of_week"] >= 5).astype(int)

display(
    df_training[
        [
            "site_id",
            "forecast_origin",
            "horizon",
            "target_timestamp",
            "target_hour",
            "target_day_of_week",
            "target_is_weekend",
            "target_consumption_kw",
        ]
    ].head(10)
)

,site_id,forecast_origin,horizon,target_timestamp,target_hour,target_day_of_week,target_is_weekend,target_consumption_kw
0,SITE001,2026-10-16 09:00:00+00:00,1,2026-10-16 10:00:00+00:00,12,4,0,174.840000
1,SITE001,2026-10-16 10:00:00+00:00,1,2026-10-16 11:00:00+00:00,13,4,0,166.120000
2,SITE001,2026-10-16 11:00:00+00:00,1,2026-10-16 12:00:00+00:00,14,4,0,169.913333
3,SITE001,2026-10-16 12:00:00+00:00,1,2026-10-16 13:00:00+00:00,15,4,0,175.741667
4,SITE001,2026-10-16 13:00:00+00:00,1,2026-10-16 14:00:00+00:00,16,4,0,175.395000
5,SITE001,2026-10-16 14:00:00+00:00,1,2026-10-16 15:00:00+00:00,17,4,0,163.173333
6,SITE001,2026-10-16 15:00:00+00:00,1,2026-10-16 16:00:00+00:00,18,4,0,166.830000
7,SITE001,2026-10-16 16:00:00+00:00,1,2026-10-16 17:00:00+00:00,19,4,0,170.803333
8,SITE001,2026-10-16 17:00:00+00:00,1,2026-10-16 18:00:00+00:00,20,4,0,128.626667
9,SITE001,2026-10-16 18:00:00+00:00,1,2026-10-16 19:00:00+00:00,21,4,0,63.660000


J - Définition des premières variables autorisées

In [128]:
feature_columns = [
    "site_id",
    "site_type",
    "horizon",
    "target_hour",
    "target_day_of_week",
    "target_month",
    "target_is_weekend",
    "lag_1h",
    "lag_24h",
    "rolling_mean_24h",
]

target_column = "target_consumption_kw"

df_model = df_training[
    [
        "forecast_origin",
        "target_timestamp",
        *feature_columns,
        target_column,
    ]
].copy()

print("Dimensions du dataset :", df_model.shape)

display(df_model.head())

Dimensions du dataset : (26292, 13)


,forecast_origin,target_timestamp,site_id,site_type,horizon,target_hour,target_day_of_week,target_month,target_is_weekend,lag_1h,lag_24h,rolling_mean_24h,target_consumption_kw
0,2026-10-16 09:00:00+00:00,2026-10-16 10:00:00+00:00,SITE001,office,1,12,4,10,0,NaN,NaN,NaN,174.840000
1,2026-10-16 10:00:00+00:00,2026-10-16 11:00:00+00:00,SITE001,office,1,13,4,10,0,NaN,NaN,NaN,166.120000
2,2026-10-16 11:00:00+00:00,2026-10-16 12:00:00+00:00,SITE001,office,1,14,4,10,0,174.840000,NaN,NaN,169.913333
3,2026-10-16 12:00:00+00:00,2026-10-16 13:00:00+00:00,SITE001,office,1,15,4,10,0,166.120000,NaN,NaN,175.741667
4,2026-10-16 13:00:00+00:00,2026-10-16 14:00:00+00:00,SITE001,office,1,16,4,10,0,169.913333,NaN,NaN,175.395000


Avant Export : Vérifications

A - Vérifications intégrité temporelle

In [129]:
assert (
    df_model["target_timestamp"]
    == df_model["forecast_origin"] + pd.to_timedelta(df_model["horizon"], unit="h")
).all()

assert df_model["horizon"].between(1, 24).all()

assert not df_model.duplicated(subset=["site_id", "forecast_origin", "horizon"]).any()

print("Intégrité temporelle validée.")

Intégrité temporelle validée.


B - Quantification valeurs manquantes

In [130]:
display(df_model.isna().sum().to_frame("missing_count"))

,missing_count
forecast_origin,0
target_timestamp,0
site_id,0
site_type,0
horizon,0
target_hour,0
target_day_of_week,0
target_month,0
target_is_weekend,0
lag_1h,336


C - Vérification des horizons disponibles

In [131]:
display(
    df_model.groupby("horizon").agg(
        observations=("target_consumption_kw", "size"),
        sites=("site_id", "nunique"),
    )
)

,observations,sites
horizon,,
1,1176,7
2,1169,7
3,1162,7
4,1155,7
5,1148,7
6,1141,7
7,1134,7
8,1127,7
9,1120,7


3 - Export des datasets

In [132]:
from pathlib import Path

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

hourly_path = OUTPUT_DIR / "hourly_dataset.parquet"
training_path = OUTPUT_DIR / "training_dataset.parquet"

df_hourly.to_parquet(hourly_path, index=False)
df_model.to_parquet(training_path, index=False)

print(f"Dataset horaire : {hourly_path}")
print(f"Dataset d'entraînement : {training_path}")

Dataset horaire : ..\data\processed\hourly_dataset.parquet
Dataset d'entraînement : ..\data\processed\training_dataset.parquet


## G — Bilan et perspectives

### Synthèse des travaux

Cette étape a permis de transformer les données énergétiques brutes issues de l'API EnerVision en deux jeux de données structurés, destinés à l'analyse et à l'entraînement de modèles prédictifs.

Le travail de préparation a été organisé autour de plusieurs opérations :

- **Validation des données sources :** vérification de la cohérence des fichiers, des périodes de collecte et des sites étudiés.
- **Contrôle qualité :** identification des valeurs manquantes, des doublons, des anomalies numériques et analyse des indicateurs de qualité.
- **Nettoyage :** définition de règles explicites permettant de préserver la fiabilité des observations sans modifier les données brutes.
- **Agrégation horaire :** transformation des mesures en séries temporelles continues, avec calcul d'indicateurs de complétude.
- **Construction des variables explicatives :** création de variables calendaires et historiques permettant de représenter les comportements énergétiques.
- **Préparation supervisée :** génération d'exemples de prévision pour les horizons H+1 à H+24, en distinguant les instants de prévision des instants cibles.

### Résultats obtenus

À partir de 7 000 observations réparties sur sept sites, nous avons constitué :

| Dataset | Volume | Finalité |
|---|---:|---|
| Dataset horaire | 1 183 observations | Référentiel analytique des consommations |
| Dataset supervisé | 26 292 exemples | Prévisions horaires sur 24 heures |

Les contrôles ont confirmé la continuité temporelle des séries horaires, l'absence de doublons et la cohérence des horizons de prédiction.

Les deux datasets ont été exportés au format Parquet dans le répertoire `data/processed/`, afin de faciliter leur réutilisation dans les étapes suivantes.

### Limites identifiées

Plusieurs limites doivent néanmoins être prises en compte.

L'historique utilisé couvre approximativement une semaine. Il permet de valider la chaîne de préparation, mais reste insuffisant pour entraîner et évaluer un modèle prédictif robuste, notamment concernant les variations saisonnières.

Certaines variables historiques présentent également des valeurs manquantes, liées aux données sources et à l'absence d'historique disponible au début des séries.

Enfin, les exemples supervisés sont construits à partir de fenêtres temporelles qui se chevauchent. Une attention particulière devra donc être accordée à la séparation chronologique des données afin d'éviter toute fuite d'information entre entraînement et validation.

### Perspectives

La prochaine phase consistera à industrialiser progressivement cette préparation et à engager les premières expérimentations de Machine Learning.

Les travaux porteront notamment sur :

- L'extraction des transformations dans un module Python de preprocessing réutilisable et testable.
- La mise en place de MLflow pour assurer la traçabilité des expérimentations.
- La définition d'une stratégie de validation chronologique adaptée aux prévisions multi-horizons.
- L'entraînement de modèles de référence et de premiers modèles prédictifs.
- La comparaison des performances à l'aide de métriques adaptées, notamment MAE et RMSE.
- L'enrichissement progressif de l'historique disponible pour améliorer la représentativité des données.

Cette étape constitue ainsi une première base analytique reproductible, sur laquelle pourra s'appuyer le futur pipeline de Machine Learning du Smart Energy Optimizer.